In [1]:
import os
from pathlib import Path

import pandas as pd
import polars as pl

import matplotlib.pyplot as plt
import seaborn as sns

import scipy
import sklearn
import tensorflow as tf
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

import mllabs

import sys
print(sys.version)

for i in [pd, pl, plt, sns, scipy, sklearn, tf, xgb, lgb, cb, mllabs]:
    if hasattr(i, '__version__'):
        print(i.__name__, i.__version__)
    else:
        print(i.__name__)

2026-03-06 01:20:00.850328: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-06 01:20:00.889589: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-06 01:20:01.942448: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/sun9sun9/python312/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: Fu

3.12.12 (main, Dec 27 2025, 11:08:36) [GCC 13.3.0]
pandas 2.3.3
polars 1.38.1
matplotlib.pyplot
seaborn 0.13.2
scipy 1.16.3
sklearn 1.8.0
tensorflow 2.20.0
xgboost 3.2.0
lightgbm
catboost 1.2.8
mllabs 0.6.0


In [5]:
data_path = Path('data')

if not os.path.exists(data_path / 'train.csv'):
    !kaggle competitions download -c playground-series-s6e3
    !unzip playground-series-s6e3.zip -d data
    !rm playground-series-s6e3.zip

 87%|█████████████████████████████████     | 13.0M/14.9M [00:01<00:00, 17.8MB/s]
100%|██████████████████████████████████████| 14.9M/14.9M [00:01<00:00, 11.2MB/s]
Archive:  playground-series-s6e3.zip
  inflating: data/sample_submission.csv  
  inflating: data/test.csv           
  inflating: data/train.csv          


In [17]:
from mllabs.processor import PolarsLoader, PandasConverter, ExprProcessor
from sklearn.pipeline import make_pipeline

loader = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    PandasConverter(index_col = 'id')
)
df_train = loader.fit_transform([data_path / 'train.csv']).assign(
    Churn = lambda x: (x['Churn'] == 'Yes').astype('int8')
)
df_test = loader.transform([data_path / 'test.csv'])

In [18]:
df_data_spec = loader[0].df_type_.join(
    pd.Series(loader[0].pl_type_, name = 'actual_dtype')
)
display(df_data_spec)
df_train.shape, df_test.shape

,min,max,na,count,n_unique,dtype,f32,i32,i16,i8,actual_dtype
feature,,,,,,,,,,,
Churn,NaN,NaN,0.0,594194.0,2.0,String,False,False,False,False,Categorical
Contract,NaN,NaN,0.0,594194.0,3.0,String,False,False,False,False,Categorical
Dependents,NaN,NaN,0.0,594194.0,2.0,String,False,False,False,False,Categorical
DeviceProtection,NaN,NaN,0.0,594194.0,3.0,String,False,False,False,False,Categorical
InternetService,NaN,NaN,0.0,594194.0,3.0,String,False,False,False,False,Categorical
MonthlyCharges,18.25,118.75,0.0,594194.0,1921.0,Float64,True,True,True,True,Float32
MultipleLines,NaN,NaN,0.0,594194.0,3.0,String,False,False,False,False,Categorical
OnlineBackup,NaN,NaN,0.0,594194.0,3.0,String,False,False,False,False,Categorical
OnlineSecurity,NaN,NaN,0.0,594194.0,3.0,String,False,False,False,False,Categorical


((594194, 20), (254655, 19))

In [19]:
set(df_train.columns) - set(df_test.columns)

{'Churn'}

In [20]:
target = 'Churn'
X_all = df_test.columns.tolist()

In [21]:
df_train[target].value_counts()

Churn
0    460377
1    133817
Name: count, dtype: int64

In [26]:
df_data_spec.query("n_unique == 2.0")

,min,max,na,count,n_unique,dtype,f32,i32,i16,i8,actual_dtype
feature,,,,,,,,,,,
Churn,NaN,NaN,0.0,594194.0,2.0,String,False,False,False,False,Categorical
Dependents,NaN,NaN,0.0,594194.0,2.0,String,False,False,False,False,Categorical
PaperlessBilling,NaN,NaN,0.0,594194.0,2.0,String,False,False,False,False,Categorical
Partner,NaN,NaN,0.0,594194.0,2.0,String,False,False,False,False,Categorical
PhoneService,NaN,NaN,0.0,594194.0,2.0,String,False,False,False,False,Categorical
SeniorCitizen,0.0,1.0,0.0,594194.0,2.0,Int64,True,True,True,True,Int8
gender,NaN,NaN,0.0,594194.0,2.0,String,False,False,False,False,Categorical


In [37]:
df_train.loc[:, df_data_spec.query("n_unique == 2.0").index].apply(
    lambda x: x.unique().tolist(), axis = 0, result_type = 'reduce'
)

feature
Churn                       [0, 1]
Dependents               [Yes, No]
PaperlessBilling         [Yes, No]
Partner                  [Yes, No]
PhoneService             [Yes, No]
SeniorCitizen               [0, 1]
gender              [Male, Female]
dtype: object

In [34]:
pd.DataFrame.apply?

Signature:
pd.DataFrame.apply(
    self,
    func: 'AggFuncType',
    axis: 'Axis' = 0,
    raw: 'bool' = False,
    result_type: "Literal['expand', 'reduce', 'broadcast'] | None" = None,
    args=(),
    by_row: "Literal[False, 'compat']" = 'compat',
    engine: "Literal['python', 'numba']" = 'python',
    engine_kwargs: 'dict[str, bool] | None' = None,
    **kwargs,
)
Docstring:
Apply a function along an axis of the DataFrame.

Objects passed to the function are Series objects whose index is
either the DataFrame's index (``axis=0``) or the DataFrame's columns
(``axis=1``). By default (``result_type=None``), the final return type
is inferred from the return type of the applied function. Otherwise,
it depends on the `result_type` argument.

Parameters
----------
func : function
    Function to apply to each column or row.
axis : {0 or 'index', 1 or 'columns'}, default 0
    Axis along which the function is applied:

    * 0 or 'index': apply function to each column.
    * 1 or 'columns